In [1]:
import torch
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader

import pandas as pd
import numpy as np

import os
import json

from utils.hake_dataset import *

from hake.models import HAKE
from hake.data import DataReader, TrainDataset, BatchType, BidirectionalOneShotIterator
from torch.utils.data import DataLoader

import warnings
warnings.filterwarnings('ignore')

/home/mai/miniconda3/envs/pyg-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = './data'

# folder to save entity id mapping
ID_MAPPING = os.path.join(DATA_PATH, 'entity_id_map')
os.makedirs(ID_MAPPING, exist_ok=True)

# folder to save edge index
EDGE_INDEX = os.path.join(DATA_PATH, 'edge_index')
os.makedirs(EDGE_INDEX, exist_ok=True)

# folder to save graph data
GRAPH_DATA = os.path.join(DATA_PATH, 'graph_data')
os.makedirs(GRAPH_DATA, exist_ok=True)

# folder to save HAKE data and model
HAKE_PATH = './hake'
os.makedirs(HAKE_PATH, exist_ok=True)

HAKE_DATA = os.path.join(HAKE_PATH, 'data/yfinance_kge')
os.makedirs(HAKE_DATA, exist_ok=True)

HAKE_EMBEDDINGS = os.path.join(HAKE_PATH, 'embeddings/yfinance_kge')
os.makedirs(HAKE_EMBEDDINGS, exist_ok=True)

# 1. Create data input for HAKE model

In [3]:
# prepare triples input
hake_triples, entity2gid = make_hake_triples(id_mapping_dir=ID_MAPPING, edge_index_dir=EDGE_INDEX,)

# save entity2gid to later map HAKE embeddings to graph
with open(os.path.join(ID_MAPPING, 'entity2gid.json'), 'w') as f:
    json.dump(entity2gid, f, indent=2)

In [4]:
# convert the triples of the dataset into a HAKE compatible dataset
make_hake_dataset(
    triples=hake_triples,
    # entity2gid=entity2gid,
    out_dir = HAKE_DATA,
)

# 2. Use HAKE to generate node embeddings

In [5]:
# !touch hake/data/yfinance_kge/valid.txt
# !touch hake/data/yfinance_kge/test.txt

In [5]:
data_reader = DataReader(HAKE_DATA)

num_entity = len(data_reader.entity_dict)
num_relation = len(data_reader.relation_dict)

print(f"Num entities: {num_entity}")
print(f"Num relations: {num_relation}")

Num entities: 20050
Num relations: 4


In [6]:
MODEL = 'HAKE'
HIDDEN_DIM = 200
GAMMA = 12.0
BATCH_SIZE = 512
NEGATIVE_SAMPLE_SIZE = 512
LEARNING_RATE = 0.0001
MAX_STEPS = 10000
MODULUS_WEIGHT = 1.0
PHASE_WEIGHT = 0.5
ADVERSARIAL_TEMPERATURE = 1.0
CPU_NUM = 4

In [7]:
# -----------------------------
# Model
# -----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = HAKE(num_entity, num_relation, HIDDEN_DIM, GAMMA, MODULUS_WEIGHT, PHASE_WEIGHT).to(device)

In [8]:
# -----------------------------
# DataLoader & iterator
# -----------------------------
train_dataloader_head = DataLoader(
    TrainDataset(data_reader, NEGATIVE_SAMPLE_SIZE, BatchType.HEAD_BATCH),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=CPU_NUM,
    collate_fn=TrainDataset.collate_fn
)

train_dataloader_tail = DataLoader(
    TrainDataset(data_reader, NEGATIVE_SAMPLE_SIZE, BatchType.TAIL_BATCH),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=CPU_NUM,
    collate_fn=TrainDataset.collate_fn
)

train_iterator = BidirectionalOneShotIterator(train_dataloader_head, train_dataloader_tail)

In [9]:
# -----------------------------
# Optimizer
# -----------------------------
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [10]:
# -----------------------------
# Training loop
# -----------------------------
for step in range(MAX_STEPS):
    log = model.train_step(model, optimizer, train_iterator, args=type('args', (), {'adversarial_temperature': ADVERSARIAL_TEMPERATURE}))
    
    if step % 200 == 0:
        print(f"Step {step}: loss={log['loss']:.4f}")

Step 0: loss=3.3815
Step 200: loss=2.0318
Step 400: loss=0.9441
Step 600: loss=0.6219
Step 800: loss=0.4923
Step 1000: loss=0.4642
Step 1200: loss=0.3881
Step 1400: loss=0.3625
Step 1600: loss=0.3149
Step 1800: loss=0.2903
Step 2000: loss=0.2798
Step 2200: loss=0.2604
Step 2400: loss=0.2542
Step 2600: loss=0.2275
Step 2800: loss=0.2238
Step 3000: loss=0.2073
Step 3200: loss=0.2017
Step 3400: loss=0.1995
Step 3600: loss=0.1808
Step 3800: loss=0.1865
Step 4000: loss=0.1606
Step 4200: loss=0.1661
Step 4400: loss=0.1464
Step 4600: loss=0.1465
Step 4800: loss=0.1456
Step 5000: loss=0.1461
Step 5200: loss=0.1509
Step 5400: loss=0.1452
Step 5600: loss=0.1461
Step 5800: loss=0.1175
Step 6000: loss=0.1323
Step 6200: loss=0.1269
Step 6400: loss=0.1344
Step 6600: loss=0.1277
Step 6800: loss=0.1222
Step 7000: loss=0.1233
Step 7200: loss=0.1255
Step 7400: loss=0.1209
Step 7600: loss=0.1151
Step 7800: loss=0.1112
Step 8000: loss=0.1179
Step 8200: loss=0.1154
Step 8400: loss=0.1074
Step 8600: loss=0.

In [11]:
# -----------------------------
# Save embeddings
# -----------------------------
torch.save(model.entity_embedding.detach().cpu(), os.path.join(HAKE_EMBEDDINGS, 'entity_embedding.pt'))
torch.save(model.relation_embedding.detach().cpu(), os.path.join(HAKE_EMBEDDINGS, 'relation_embedding.pt'))
print("Embeddings saved!")

Embeddings saved!


# 3. Add HAKE embeddings to the hetero graph created
Hake embeddings for entities uses the index in the `entities.dict` file so we'll have to map it to our original id mapping

In [13]:
# Load HAKE embeddings (shape [num_entities, dim])
emb = torch.load(os.path.join(HAKE_EMBEDDINGS, 'entity_embedding.pt')).cpu().numpy()

dim = emb.shape[1]

# load entity name: global id mapping of the heterogeneous graph
with open(os.path.join(ID_MAPPING, 'entity2gid.json')) as f:
    entity2gid = json.load(f)

num_global = max(entity2gid.values()) + 1

# load entities.dict created for HAKE embedding
def load_entities_dict(path):
    entity2id = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            eid, name = line.strip().split("\t")
            entity2id[name] = int(eid)
    return entity2id

hake_entities = load_entities_dict(os.path.join(HAKE_DATA, 'entities.dict'))

# Your final embedding matrix aligned to the graph
gid_emb = np.zeros((num_global, dim))

for name, gid in entity2gid.items():
    hake_id = hake_entities[name]  # from entities.dict OR your code
    gid_emb[gid] = emb[hake_id]


In [14]:
gid_emb.shape

(20161, 400)

In [15]:
gid_emb

array([[ 0.01394556, -0.01094944, -0.02084748, ...,  0.07957357,
        -0.1119857 ,  0.11726134],
       [ 0.08299417, -0.0082517 ,  0.06807277, ..., -0.06980586,
         0.03736908, -0.08350236],
       [-0.0389362 , -0.01699718,  0.04030496, ...,  0.08321684,
         0.0164444 , -0.09648882],
       ...,
       [-0.00481487,  0.02522892, -0.03405559, ...,  0.08532265,
         0.13108146,  0.14204949],
       [-0.02554752, -0.00383731, -0.04241296, ..., -0.09573608,
        -0.0546038 , -0.02329703],
       [-0.02368834, -0.00345524, -0.03459214, ..., -0.08889662,
         0.12827355,  0.12770458]])

In [20]:
# load heterogeneous graph created so far
data = torch.load(os.path.join(GRAPH_DATA, 'yfinance_kge.pt'), weights_only=False)

# convert gid_emb to torch tensor
gid_emb_tensor = torch.tensor(gid_emb, dtype=torch.float)

# load node id offsets
with open(os.path.join(ID_MAPPING, 'global_entity_id_offsets.json')) as f:
    offset_start = json.load(f)

# compute the start and end id of each entity types
sorted_types = sorted(offset_start.items(), key=lambda x: x[1])

offsets = {}

for i, (ntype, start) in enumerate(sorted_types):
    if i < len(sorted_types) - 1:
        _, next_start = sorted_types[i+1]
        end = next_start
    else:
        end = gid_emb.shape[0]
    offsets[ntype] = (start, end)

# Now concat HAKE embeddings to the node features in HeteroData
gid_emb_tensor = torch.tensor(gid_emb, dtype=torch.float)

for ntype, (start, end) in offsets.items():
    node_emb = gid_emb_tensor[start:end]
    
    if 'x' in data[ntype]:
        data[ntype].x = torch.cat([data[ntype].x, node_emb], dim=1)
    else:
        data[ntype].x = node_emb

    print(f"{ntype}: {data[ntype].x.shape}")

company: torch.Size([4164, 438])
stock_symbol: torch.Size([4607, 400])
industry: torch.Size([146, 409])
sector: torch.Size([13, 407])
institution: torch.Size([2669, 400])
fund: torch.Size([2222, 400])
fund_symbol: torch.Size([6340, 429])


In [21]:
data

HeteroData(
  company={ x=[4164, 438] },
  fund_symbol={ x=[6340, 429] },
  sector={ x=[13, 407] },
  industry={ x=[146, 409] },
  stock_symbol={ x=[4607, 400] },
  institution={ x=[2669, 400] },
  fund={ x=[2222, 400] },
  (company, has_symbol, stock_symbol)={ edge_index=[2, 4508] },
  (company, belongs_to, industry)={ edge_index=[2, 3992] },
  (industry, is_part_of, sector)={ edge_index=[2, 144] },
  (institution, holds, stock_symbol)={ edge_index=[2, 39512] },
  (fund, holds, stock_symbol)={ edge_index=[2, 37098] },
  (fund, has_symbol, fund_symbol)={ edge_index=[2, 6851] }
)

In [22]:
# save graph with KGE embeddings and original features
torch.save(data, os.path.join(GRAPH_DATA, 'yfinance_kge_full.pt'))